# Lecture 12: Graph Kernels — Exercises

Companion to the note [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).

Practise kernels on structured data: Mercer's PSD condition, Haussler's R-convolution, binary and $n$-spectrum string kernels, graphlet counts (labelled vs unlabelled, automorphisms), the random-walk kernel on the direct product graph, and the Weisfeiler–Lehman subtree kernel, including its famous failure case ($C_3\cup C_3$ vs $C_6$), the depth $h$, cosine normalisation and a kernel SVM on a synthetic graph classification task. NumPy + scikit-learn only (no networkx).

**17 exercises** · 🧠 Concept ×5 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

# Zachary's Karate Club (same 0-indexed edge list as networkx.karate_club_graph(), 34 nodes, 78 edges)
KARATE_EDGES = [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 10), (0, 11), (0, 12), (0, 13),
    (0, 17), (0, 19), (0, 21), (0, 31), (1, 2), (1, 3), (1, 7), (1, 13), (1, 17), (1, 19), (1, 21), (1, 30), (2, 3),
    (2, 7), (2, 8), (2, 9), (2, 13), (2, 27), (2, 28), (2, 32), (3, 7), (3, 12), (3, 13), (4, 6), (4, 10), (5, 6),
    (5, 10), (5, 16), (6, 16), (8, 30), (8, 32), (8, 33), (9, 33), (13, 33), (14, 32), (14, 33), (15, 32), (15, 33),
    (18, 32), (18, 33), (19, 33), (20, 32), (20, 33), (22, 32), (22, 33), (23, 25), (23, 27), (23, 29), (23, 32),
    (23, 33), (24, 25), (24, 27), (24, 31), (25, 31), (26, 29), (26, 33), (27, 33), (28, 31), (28, 33), (29, 32),
    (29, 33), (30, 32), (30, 33), (31, 32), (31, 33), (32, 33)]
# Ground-truth split after the club broke up: 0 = "Mr. Hi", 1 = "Officer"
KARATE_CLUB = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

def edges_to_adj(edges, n=None, directed=False):
    """Edge list -> dense 0/1 NumPy adjacency matrix."""
    n = n if n is not None else max(max(e) for e in edges) + 1
    A = np.zeros((n, n))
    for u, v in edges:
        A[u, v] = 1
        if not directed:
            A[v, u] = 1
    return A

A_karate = edges_to_adj(KARATE_EDGES, 34)

import itertools
from collections import Counter
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold
rng = np.random.default_rng(12)

def cycle_graph(n):
    return edges_to_adj([(i, (i + 1) % n) for i in range(n)], n)

def disjoint_union(*As):
    n = sum(len(A) for A in As); U = np.zeros((n, n)); o = 0
    for A in As:
        U[o:o + len(A), o:o + len(A)] = A; o += len(A)
    return U

C3, C6 = cycle_graph(3), cycle_graph(6)
two_triangles = disjoint_union(C3, C3)
P4 = edges_to_adj([(0, 1), (1, 2), (2, 3)], 4)
STAR = edges_to_adj([(0, 1), (0, 2), (0, 3)], 4)
PAW = edges_to_adj([(0, 1), (1, 2), (0, 2), (2, 3)], 4)   # triangle with a pendant node

## Part A · Concepts

### Exercise 1 · Why kernels, and why PSD?
*🧠 Concept · ★☆☆*

1. Why can't we feed graphs of different sizes straight into an SVM, and how does a graph kernel get around this?
2. State Mercer's condition. Why does an SVM need the Gram matrix to be PSD (think about the dual problem)?
3. Is $K(G, G') = -|\,|V| - |V'|\,|$ a valid kernel? (Hint: test a $2\times2$ Gram matrix.)

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The SVM dual maximises $\sum_i\alpha_i - \tfrac12\sum_{ij}\alpha_i\alpha_jy_iy_jK_{ij}$; for that to be a concave problem…

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Graphs have no fixed-length coordinates and no canonical node order, so there is no natural vector. A kernel only needs a similarity
   $K(G,G') = \langle\phi(G),\phi(G')\rangle$, where $\phi$ may live in a huge (even infinite-dimensional) space of substructure counts; the SVM
   only ever touches the Gram matrix.
2. $K$ is a valid kernel iff it is symmetric and every Gram matrix is PSD: $\sum_{ij}c_ic_jK(x_i,x_j)\ge0$. PSD makes the dual quadratic
   form $\tfrac12\alpha^\top(yy^\top\circ K)\alpha$ convex, so the dual is a concave maximisation with a global optimum, and guarantees that a feature map $\phi$ exists.
3. No. For two graphs with sizes differing by 1: $K = \begin{pmatrix}0&-1\\-1&0\end{pmatrix}$ has eigenvalues $\pm1$, so it's not PSD.

</details>

### Exercise 2 · R-convolution instances
*🧠 Concept · ★★☆*

In Haussler's framework $K_D(d,d') = \sum_{p\in\mathcal R^{-1}(d)}\sum_{p'\in\mathcal R^{-1}(d')}K_P(p,p')$, identify $\mathcal P$, $\mathcal R$ and $K_P$
for: (a) the ordinary dot product on $\mathbb R^d$, (b) the binary $n$-gram kernel, (c) the $n$-spectrum kernel, (d) the graphlet kernel,
(e) the WL subtree kernel. Why does the $n$-spectrum version count *repeated* occurrences while the binary version doesn't?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

For (c) the parts are (position, $n$-gram) pairs, so two occurrences at different positions are different parts.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\mathcal P$ = (index, value) pairs $(i, x[i])$; $K_P((i,y),(j,z)) = yz\,\mathbb 1[i=j]$ → $\sum_i x_ix'_i$.
(b) $\mathcal P = \Sigma^n$, $(u,s)\in\mathcal R$ iff $u$ occurs in $s$ (a **set** of distinct $n$-grams), $K_P = \mathbb 1[u=v]$ → number of shared distinct $n$-grams.
(c) $\mathcal P = \mathbb N\times\Sigma^n$, parts are $(k,u)$ = "$u$ starts at position $k$"; $K_P((k,u),(k',v)) = \mathbb 1[u=v]$ (position ignored) →
    $\sum_u\text{count}_u(s)\text{count}_u(t)$. Repeats are distinct parts because their positions differ.
(d) $\mathcal P$ = induced $k$-node subgraphs; $K_P$ = 1 if isomorphic to the same graphlet → $\langle\phi(G),\phi(G')\rangle$ of graphlet counts.
(e) $\mathcal P$ = (node, iteration) pairs, i.e. rooted subtrees of depth $t$; $K_P$ = 1 if the WL colours (subtree hashes) are equal.

</details>

### Exercise 3 · When 1-WL fails
*🧠 Concept · ★★☆*

1. Run 1-WL mentally on $C_3\cup C_3$ and $C_6$. Why are the colour histograms identical in every round?
2. Name a simple graph statistic that *does* distinguish them.
3. Message-passing GNNs (Lectures 10–11) can't be more powerful than 1-WL at distinguishing graphs. Explain why, and what extra condition
   makes a GNN *as* powerful as 1-WL.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Both graphs are 2-regular. A GNN layer is an aggregate-then-update map; when is it injective on multisets?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. All 6 nodes start with colour 1. Every node has exactly two neighbours of colour 1, so every node gets the same new colour, round after round:
   the partition never splits, and both histograms are $\{c_t: 6\}$ for every $t$.
2. Triangle count (2 vs 0), connectivity (2 components vs 1), the diameter, or the spectrum of $A$ (eigenvalue 2 has multiplicity 2 vs 1).
   Graphlet kernels see the triangles immediately.
3. A GNN layer computes $h_v' = \text{UPDATE}(h_v, \text{AGG}(\{\!\{h_u\}\!\}))$, which is a function of exactly the information WL hashes. If WL
   gives two nodes the same colour, any GNN gives them the same embedding (by induction over layers). A GNN matches 1-WL when AGG and UPDATE are
   **injective on multisets**, e.g. sum aggregation followed by an MLP (GIN). Mean or max aggregation is strictly weaker.

</details>

### Exercise 4 · Depth, normalisation and cost
*🧠 Concept · ★★☆*

1. Explain the NCI1 curve (≈75% at $h=1$, ≈85% at $h=3$, ≈81% at $h=5$) in terms of under- and over-fitting of the WL features.
2. Why normalise $K(G,G')/\sqrt{K(G,G)K(G',G')}$, and what is the diagonal afterwards?
3. Order the graphlet ($k$ fixed), random-walk and WL kernels by the cost of one kernel evaluation on graphs with $n$ nodes and $m$ edges.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Deep WL colours are rooted subtrees of depth $h$: how many graphs share an exact depth-5 subtree?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Small $h$: colours are only atom type + degree, too coarse to recognise functional groups (under-fitting). $h\approx3$: subtrees
   describe rings and chains that matter chemically. Large $h$: subtrees are nearly unique to each molecule, so off-diagonal similarities
   shrink towards the shared low-$h$ part, the Gram matrix approaches a diagonal, and the SVM memorises (over-fitting).
2. Raw counts grow with graph size, so big graphs look similar to everything. Normalisation is the cosine of the angle between feature vectors:
   the diagonal becomes exactly 1 and values lie in $[0,1]$ for count features. It's still PSD ($D^{-1/2}KD^{-1/2}$).
3. WL: $O(hm)$ per graph to compute features, which are then reused for all pairs (linear!). Graphlets: $O(n^k)$ subsets per graph (with a cheap
   isomorphism test for small $k$). Random walk: $O(n^6)$ naively for the $(n^2)\times(n^2)$ inversion, $O(n^3)$–$O(n^4)$ with tricks (the note's figure).
   WL is by far the cheapest, which together with its accuracy explains its popularity.

</details>

### Exercise 5 · Graphlets: counting and complexity
*🧠 Concept · ★☆☆*

1. Why are only **connected** graphlets used in the note's $\mathcal G_3 = \{P_3, K_3\}$, and what's lost by ignoring the disconnected 3-node
   patterns?
2. For $k = 5$ on a graph with 1000 nodes, how many vertex subsets must be inspected? What's the usual remedy?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\binom{1000}{5}$ is large.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Disconnected patterns (an edge plus an isolated node, three isolated nodes) are dominated by "random" combinations of far-apart nodes and
   their counts are determined by lower-order statistics ($n$, $m$, and the connected counts), so they add little and inflate the vectors.
   Some definitions keep them; the note's version doesn't.
2. $\binom{1000}{5}\approx8.25\times10^{12}$ subsets, each needing an isomorphism test: infeasible. Remedies: **sample** random $k$-subsets and
   estimate the graphlet distribution (with Hoeffding-style bounds), or use clever enumeration for $k\le5$ in bounded-degree graphs.

</details>

## Part B · By hand

### Exercise 6 · String kernels by hand
*✍️ By hand · ★☆☆*

$s = \texttt{abab}$, $t = \texttt{baba}$, $n = 2$. Compute the binary occurrence kernel $K^{(1)}(s,t)$, the 2-spectrum kernel $K^{(2)}(s,t)$ and
$K^{(2)}(s,s)$.

In [ ]:
k_bin_st = None
k_spec_st = None
k_spec_ss = None

_c = lambda w: Counter(w[i:i + 2] for i in range(len(w) - 1))
check('binary kernel', k_bin_st, len(set(_c('abab')) & set(_c('baba'))))
check('spectrum kernel', k_spec_st, sum(_c('abab')[u] * _c('baba')[u] for u in _c('abab')))
check('spectrum K(s,s)', k_spec_ss, sum(v * v for v in _c('abab').values()))

<details>
<summary><b>💡 Hint</b></summary>

2-grams of abab: ab, ba, ab. Of baba: ba, ab, ba.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Distinct 2-grams: $s$: $\{ab, ba\}$, $t$: $\{ab, ba\}$ → $K^{(1)} = 2$.
Counts: $s$: $ab{:}2, ba{:}1$; $t$: $ab{:}1, ba{:}2$ → $K^{(2)}(s,t) = 2\cdot1 + 1\cdot2 = 4$; $K^{(2)}(s,s) = 4 + 1 = 5$.
Normalised: $4/\sqrt{5\cdot5} = 0.8$, whereas the binary kernel normalised gives $2/2 = 1$: binary can't tell the strings apart.

```python
k_bin_st = 2
k_spec_st = 4
k_spec_ss = 5
```

</details>

### Exercise 7 · Graphlet counts of the paw graph
*✍️ By hand · ★☆☆*

The *paw*: triangle $0,1,2$ plus a pendant node 3 attached to 2. List all $\binom43 = 4$ vertex triples, classify each induced subgraph
(path $P_3$, triangle $K_3$, or disconnected), and give $\phi^{(2)}$ (unlabelled) and $\phi^{(1)} = |\text{Aut}(g)|\cdot\phi^{(2)}$ (labelled) for
$(P_3, K_3)$.

In [ ]:
phi2_paw = None   # [#P3, #K3]
phi1_paw = None

_c = [0, 0]
for _T in itertools.combinations(range(4), 3):
    _e = PAW[np.ix_(_T, _T)].sum() / 2
    if _e == 2: _c[0] += 1
    elif _e == 3: _c[1] += 1
check('unlabelled counts', phi2_paw, _c); check('labelled counts', phi1_paw, [2 * _c[0], 6 * _c[1]])

<details>
<summary><b>💡 Hint</b></summary>

A 3-node induced subgraph with 2 edges is always a path; with 3 edges a triangle; with $\le 1$ edge it's disconnected.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\{0,1,2\}$: triangle. $\{0,1,3\}$: only edge $0$–$1$ → disconnected. $\{0,2,3\}$: $0$–$2$–$3$ → path. $\{1,2,3\}$: path.
$\phi^{(2)} = (2, 1)$, $\phi^{(1)} = (2\cdot2, 6\cdot1) = (4, 6)$.

```python
phi2_paw = [2, 1]
phi1_paw = [4, 6]
```

</details>

### Exercise 8 · Counting automorphisms
*✍️ By hand · ★★☆*

Give $|\text{Aut}(g)|$ (number of node permutations that map the edge set onto itself) for: $P_3$, $K_3$, the star $K_{1,3}$, the path $P_4$,
the 4-cycle $C_4$. Then explain why $\phi^{(1)}_g(G) = |\text{Aut}(g)|\,\phi^{(2)}_g(G)$.

In [ ]:
aut = None   # dict: {'P3': ..., 'K3': ..., 'star': ..., 'P4': ..., 'C4': ...}

def _aut(A):
    n = len(A); return sum(1 for p in itertools.permutations(range(n)) if np.array_equal(A[np.ix_(p, p)], A))
_ref = {'P3': _aut(edges_to_adj([(0, 1), (1, 2)], 3)), 'K3': _aut(C3), 'star': _aut(STAR), 'P4': _aut(P4), 'C4': _aut(cycle_graph(4))}
if aut is not None:
    for _k in _ref: check(f'|Aut({_k})|', aut.get(_k), _ref[_k])
else:
    check('automorphisms', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

$C_4$: rotations and reflections of a square (dihedral group). Star: the centre is fixed, leaves can be permuted freely.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P_3$: 2 (swap the ends). $K_3$: $3! = 6$. Star $K_{1,3}$: $3! = 6$ (permute leaves). $P_4$: 2 (reverse). $C_4$: 8 (4 rotations × 2 reflections).

Every induced copy (a vertex set $V'$ with $G[V']\simeq g$) admits exactly $|\text{Aut}(g)|$ distinct isomorphisms $f: V_g\to V'$: fix one,
then all others are it composed with an automorphism of $g$. So counting maps counts each vertex set $|\text{Aut}(g)|$ times.

```python
aut = {'P3': 2, 'K3': 6, 'star': 6, 'P4': 2, 'C4': 8}
```

</details>

### Exercise 9 · One WL round by hand
*✍️ By hand · ★★☆*

Unlabelled graphs $P_4$ ($0-1-2-3$) and the star $K_{1,3}$. Run one WL iteration ($h=1$) starting from colour 1 everywhere. Write the
colour histograms for $t = 0, 1$ and compute $K^{(1)}_{WL}(P_4, \text{star})$, $K^{(1)}_{WL}(P_4, P_4)$, $K^{(1)}_{WL}(\text{star},\text{star})$ and
the normalised kernel.

In [ ]:
k_wl_ps = None
k_wl_pp = None
k_wl_ss = None
k_wl_norm = None

def _wl_hist(A, h):
    col = [('1',)] * len(A); hist = Counter((0, c) for c in col)
    for t in range(1, h + 1):
        col = [(col[v], tuple(sorted(col[u] for u in np.flatnonzero(A[v])))) for v in range(len(A))]
        hist.update((t, c) for c in col)
    return hist
_hp, _hs = _wl_hist(P4, 1), _wl_hist(STAR, 1)
_k = lambda a, b: sum(a[x] * b.get(x, 0) for x in a)
check('K(P4, star)', k_wl_ps, _k(_hp, _hs)); check('K(P4, P4)', k_wl_pp, _k(_hp, _hp)); check('K(star, star)', k_wl_ss, _k(_hs, _hs))
check('normalised', k_wl_norm, _k(_hp, _hs) / np.sqrt(_k(_hp, _hp) * _k(_hs, _hs)), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

New colours: $a = (1,\{1\})$ for degree-1 nodes, $b = (1,\{1,1\})$, $c = (1, \{1,1,1\})$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$t=0$: both graphs $\{1: 4\}$. $t = 1$: $P_4$: $\{a: 2, b: 2\}$; star: $\{a: 3, c: 1\}$.
$K(P_4,\text{star}) = 4\cdot4 + 2\cdot3 = 22$; $K(P_4,P_4) = 16 + 4 + 4 = 24$; $K(\text{star},\text{star}) = 16 + 9 + 1 = 26$.
Normalised: $22/\sqrt{24\cdot26} = 0.8807$. Without $t\ge1$ every pair of 4-node graphs would look identical (similarity 1).

```python
k_wl_ps = 22
k_wl_pp = 24
k_wl_ss = 26
k_wl_norm = 22 / np.sqrt(24 * 26)
```

</details>

### Exercise 10 · Random-walk kernel on the product graph
*✍️ By hand · ★★☆*

For unlabelled graphs the product graph has adjacency $A_\times = A\otimes A'$ (Kronecker product). Let $G = G' = K_2$ (a single edge).

1. Write $A_\times$ ($4\times4$) and its eigenvalues.
2. Compute $K(G,G') = \mathbf 1^\top(I - \lambda A_\times)^{-1}\mathbf 1$ for $\lambda = 0.5$. Which values of $\lambda$ are allowed?

In [ ]:
k_rw_hand = None
lambda_max_allowed = None   # sup of allowed lambda

_Ax = np.kron(edges_to_adj([(0, 1)], 2), edges_to_adj([(0, 1)], 2))
check('random-walk kernel', k_rw_hand, np.ones(4) @ np.linalg.solve(np.eye(4) - 0.5 * _Ax, np.ones(4)))
check('lambda bound', lambda_max_allowed, 1 / np.max(np.abs(np.linalg.eigvalsh(_Ax))))

<details>
<summary><b>💡 Hint</b></summary>

$A_\times\mathbf 1 = \mathbf 1$ (every product node has exactly one neighbour), so $\mathbf 1$ is an eigenvector: $(I-\lambda A_\times)^{-1}\mathbf 1 = \frac{1}{1-\lambda}\mathbf 1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$A_\times = \begin{pmatrix}0&0&0&1\\0&0&1&0\\0&1&0&0\\1&0&0&0\end{pmatrix}$ (the product of two edges is two disjoint edges), eigenvalues $\pm1$.
$K = \frac{4}{1-0.5} = 8$. Equivalently $\sum_l\lambda^l\mathbf 1^\top A_\times^l\mathbf 1 = 4\sum_l 0.5^l = 8$: a common walk of every length exists.
The geometric series converges iff $\lambda < 1/\rho(A_\times) = 1$. In general $\rho(A\otimes A') = \rho(A)\rho(A')$, so $\lambda$ must shrink for denser graphs.

```python
k_rw_hand = 8.0
lambda_max_allowed = 1.0
```

</details>

### Exercise 11 · Which matrices are valid Gram matrices?
*✍️ By hand · ★☆☆*

Which of these symmetric matrices can be Gram matrices of a valid kernel? Give the smallest eigenvalue of each.

$K_a = \begin{pmatrix}1&2\\2&1\end{pmatrix}$, $K_b = \begin{pmatrix}2&1\\1&2\end{pmatrix}$, $K_c = K_b\circ K_b$ (element-wise product),
$K_d = $ the cosine normalisation of $K_b$.

In [ ]:
min_eigs = None   # [lam_a, lam_b, lam_c, lam_d]

_Kb = np.array([[2, 1], [1, 2.]]); _Kd = _Kb / np.sqrt(np.outer(np.diag(_Kb), np.diag(_Kb)))
check('smallest eigenvalues', min_eigs, [np.linalg.eigvalsh(M)[0] for M in (np.array([[1, 2], [2, 1.]]), _Kb, _Kb * _Kb, _Kd)])

<details>
<summary><b>💡 Hint</b></summary>

For $\begin{pmatrix}a&b\\b&a\end{pmatrix}$ the eigenvalues are $a\pm b$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$K_a$: $1\pm2$ → $-1$, **not** PSD (the off-diagonal similarity exceeds the self-similarities, so no feature map can do that).
$K_b$: $2\pm1$ → $1$ ✔. $K_c = \begin{pmatrix}4&1\\1&4\end{pmatrix}$ → $3$ ✔ (Schur product theorem: products of kernels are kernels).
$K_d = \begin{pmatrix}1&0.5\\0.5&1\end{pmatrix}$ → $0.5$ ✔ (congruence $D^{-1/2}KD^{-1/2}$ preserves PSD).

```python
min_eigs = [-1, 1, 3, 0.5]
```

</details>

## Part C · Code from scratch

### Exercise 12 · String kernels and their Gram matrix
*💻 Code · ★☆☆*

Implement `binary_kernel(s, t, n)` and `spectrum_kernel(s, t, n)`, then build the $5\times5$ Gram matrices of both kernels on `seqs`
($n = 3$). Verify numerically that both are PSD (`min_eig_bin`, `min_eig_spec`).

In [ ]:
seqs = ['GATTACA', 'GATTAGA', 'ACGTACGT', 'TTTTTTT', 'CAGATTAC']

def binary_kernel(s, t, n):
    return None

def spectrum_kernel(s, t, n):
    return None

K_bin = None
K_spec = None
min_eig_bin = None
min_eig_spec = None

_grams = lambda w: [w[i:i + 3] for i in range(len(w) - 2)]
_vocab = sorted({g for w in seqs for g in _grams(w)})
_Phi_b = np.array([[g in set(_grams(w)) for g in _vocab] for w in seqs], float)
_Phi_s = np.array([[_grams(w).count(g) for g in _vocab] for w in seqs], float)
check('binary Gram = Phi Phi^T', K_bin, _Phi_b @ _Phi_b.T); check('spectrum Gram = Phi Phi^T', K_spec, _Phi_s @ _Phi_s.T)
check('binary PSD', None if min_eig_bin is None else min_eig_bin > -1e-9, True)
check('spectrum PSD', None if min_eig_spec is None else min_eig_spec > -1e-9, True)

<details>
<summary><b>💡 Hint</b></summary>

`Counter(s[i:i+n] for i in range(len(s)-n+1))`; the binary kernel is `len(set_s & set_t)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Both Gram matrices equal $\Phi\Phi^\top$ for explicit count / indicator feature vectors, so they are PSD by construction:
$c^\top\Phi\Phi^\top c = \lVert\Phi^\top c\rVert^2\ge0$. `TTTTTTT` is where the two differ most: one distinct 3-gram occurring 5 times, so
$K_{spec}(\texttt{TTTTTTT},\texttt{TTTTTTT}) = 25$ but $K_{bin} = 1$.

```python
seqs = ['GATTACA', 'GATTAGA', 'ACGTACGT', 'TTTTTTT', 'CAGATTAC']

def binary_kernel(s, t, n):
    return len({s[i:i + n] for i in range(len(s) - n + 1)} & {t[i:i + n] for i in range(len(t) - n + 1)})

def spectrum_kernel(s, t, n):
    cs = Counter(s[i:i + n] for i in range(len(s) - n + 1))
    ct = Counter(t[i:i + n] for i in range(len(t) - n + 1))
    return sum(c * ct[u] for u, c in cs.items())

K_bin = np.array([[binary_kernel(a, b, 3) for b in seqs] for a in seqs], float)
K_spec = np.array([[spectrum_kernel(a, b, 3) for b in seqs] for a in seqs], float)
min_eig_bin = np.linalg.eigvalsh(K_bin)[0]
min_eig_spec = np.linalg.eigvalsh(K_spec)[0]
print(K_spec)
```

</details>

### Exercise 13 · Size-3 graphlet features
*💻 Code · ★★☆*

Implement `graphlet3(A)` returning the unlabelled counts $\phi^{(2)} = (\#P_3, \#K_3)$ of **induced** 3-node subgraphs by enumerating all
vertex triples with `itertools.combinations`. Run it on karate (`phi_karate`). Then derive the closed forms
$\#K_3 = \tfrac16\mathrm{tr}(A^3)$ and $\#P_3^{\text{induced}} = \sum_v\binom{d_v}{2} - 3\,\#K_3$ and explain the $-3$.

In [ ]:
def graphlet3(A):
    # TODO: return np.array([n_paths, n_triangles])
    return None

phi_karate = graphlet3(A_karate)

_tri = np.trace(np.linalg.matrix_power(A_karate, 3)) / 6
_d = A_karate.sum(1)
check('karate graphlet counts', phi_karate, [np.sum(_d * (_d - 1) / 2) - 3 * _tri, _tri])
check('paw graph', graphlet3(PAW), [2, 1])

<details>
<summary><b>💡 Hint</b></summary>

For a triple, the number of edges in the induced subgraph is `A[np.ix_(T, T)].sum() / 2`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathrm{tr}(A^3)$ counts closed walks of length 3; each triangle gives 6 (3 starting points × 2 directions). $\sum_v\binom{d_v}{2}$ counts
all "wedges" (pairs of edges sharing a centre), i.e. *non-induced* paths of length 2. Each triangle contains 3 wedges whose induced subgraph is
the triangle, not a path, hence the $-3\#K_3$. Karate: 45 triangles, 393 induced paths. The brute force costs $\binom{34}{3} = 5984$ triples here,
but $O(n^3)$ in general, versus one matrix power for the closed form.

```python
def graphlet3(A):
    counts = np.zeros(2)
    for T in itertools.combinations(range(len(A)), 3):
        e = A[np.ix_(T, T)].sum() / 2
        if e == 2:
            counts[0] += 1
        elif e == 3:
            counts[1] += 1
    return counts

phi_karate = graphlet3(A_karate)
print(phi_karate)
```

</details>

### Exercise 14 · The Weisfeiler–Lehman subtree kernel
*💻 Code · ★★☆*

Implement `wl_features(graphs, h)`: run 1-WL on all graphs **with a shared colour dictionary** (the injective hash $H$: map each new
signature `(own colour, sorted neighbour colours)` to a fresh integer, consistently across graphs), and return a feature matrix
$\Phi\in\mathbb N^{N\times|\mathcal C|}$ counting each colour of each iteration $t = 0..h$. Then `wl_kernel(graphs, h, normalize)` $=\Phi\Phi^\top$
(optionally cosine-normalised).

Check it on `[P4, STAR]` against Exercise 9 and on `[two_triangles, C6]`: store their normalised similarity for $h = 3$ in `wl_fail`.

In [ ]:
def wl_features(graphs, h, labels=None):
    # labels: optional list of initial node-label arrays (default: all ones)
    # TODO: return Phi (N x number_of_colours)
    return None

def wl_kernel(graphs, h, normalize=False, labels=None):
    # TODO
    return None

K_ps = wl_kernel([P4, STAR], 1)
wl_fail = None

check('WL kernel P4 / star (Exercise 9)', K_ps, [[24, 22], [22, 26]])
check('1-WL cannot separate C3+C3 from C6', wl_fail, 1.0)
if K_ps is not None:
    _Kn = wl_kernel([P4, STAR, PAW, C6, two_triangles, A_karate], 3, normalize=True)
    check('normalised diagonal is 1', np.diag(_Kn), np.ones(6))
    check('normalised Gram is PSD', np.linalg.eigvalsh(_Kn)[0] > -1e-9, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Keep `col` as a list of ints per graph. Each round: `sig = (col[v], tuple(sorted(col[u] for u in nbrs(v))))`; `if sig not in table: table[sig] = len(table)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Collect `Counter`s of `(t, colour)` per graph, then build one column per distinct `(t, colour)` key.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The shared dictionary is what makes colours comparable *across* graphs; hashing each graph separately would give meaningless overlaps.
$C_3\cup C_3$ and $C_6$ get identical feature vectors at every depth, so their normalised similarity is exactly 1: the 1-WL blind spot from
Critical Insight 4.1. Cost: $O(h\,m)$ per graph for the refinement, then sparse dot products.

```python
def wl_features(graphs, h, labels=None):
    table = {}
    hists = []
    for gi, A in enumerate(graphs):
        n = len(A)
        init = labels[gi] if labels is not None else [1] * n
        col = []
        for l in init:
            key = ('init', l)
            if key not in table: table[key] = len(table)
            col.append(table[key])
        hist = Counter((0, c) for c in col)
        nbrs = [np.flatnonzero(A[v]) for v in range(n)]
        for t in range(1, h + 1):
            new = []
            for v in range(n):
                sig = (col[v], tuple(sorted(col[u] for u in nbrs[v])))
                if sig not in table: table[sig] = len(table)
                new.append(table[sig])
            col = new
            hist.update((t, c) for c in col)
        hists.append(hist)
    keys = sorted({k for hst in hists for k in hst})
    idx = {k: i for i, k in enumerate(keys)}
    Phi = np.zeros((len(graphs), len(keys)))
    for gi, hst in enumerate(hists):
        for k, c in hst.items():
            Phi[gi, idx[k]] = c
    return Phi

def wl_kernel(graphs, h, normalize=False, labels=None):
    Phi = wl_features(graphs, h, labels)
    K = Phi @ Phi.T
    if normalize:
        d = np.sqrt(np.diag(K)); K = K / np.outer(d, d)
    return K

K_ps = wl_kernel([P4, STAR], 1)
wl_fail = wl_kernel([two_triangles, C6], 3, normalize=True)[0, 1]
print(K_ps, wl_fail)
```

</details>

### Exercise 15 · Random-walk kernel with node labels
*💻 Code · ★★★*

Implement `rw_kernel(A1, A2, lam, l1=None, l2=None)` $= \mathbf 1^\top(I - \lambda A_\times)^{-1}\mathbf 1$ on the direct product graph. With node labels,
product node $(i,i')$ exists only if $l_1(i) = l_2(i')$: build $A_\times = A_1\otimes A_2$ and zero out rows/columns of non-matching pairs.
Raise a `ValueError` if $\lambda\ge1/\rho(A_\times)$.

1. Verify against the truncated series $\sum_{l=0}^{L}\lambda^l\mathbf 1^\top A_\times^l\mathbf 1$ for `PAW` vs `P4` with $\lambda = 0.1$ (`rw_paw_p4`).
2. Compute the normalised random-walk kernel of $C_3\cup C_3$ vs $C_6$ (`rw_fail`). Is it 1, like WL?

In [ ]:
def rw_kernel(A1, A2, lam, l1=None, l2=None):
    # TODO
    return None

rw_paw_p4 = rw_kernel(PAW, P4, 0.1)
rw_fail = None

_Ax = np.kron(PAW, P4); _s = 0; _v = np.ones(16)
for _l in range(200):
    _s += 0.1**_l * _v.sum(); _v = _Ax @ _v
check('RW kernel vs truncated series', rw_paw_p4, _s, tol=1e-8)
_lab1 = np.array([0, 0, 1, 1]); _lab2 = np.array([0, 1, 1, 0])
_m = (_lab1[:, None] == _lab2[None, :]).ravel()
_Axl = np.kron(PAW, P4) * np.outer(_m, _m)
check('labelled RW kernel', rw_kernel(PAW, P4, 0.1, _lab1, _lab2), (np.ones(16) * _m) @ np.linalg.solve(np.eye(16) - 0.1 * _Axl, np.ones(16) * _m))
check('C3+C3 vs C6 (both 2-regular): random walks cannot tell them apart either', rw_fail, 1.0)

<details>
<summary><b>💡 Hint 1</b></summary>

Product nodes that don't match get no edges **and** should not count as start/end points: use $\mathbf m$ (the 0/1 match mask) instead of $\mathbf 1$ in $\mathbf m^\top(I-\lambda A_\times)^{-1}\mathbf m$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Normalisation: `K12 / np.sqrt(K11 * K22)`; for the 2-regular graphs pick $\lambda = 0.1 < 1/4$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

In a $d$-regular graph every node has the same number of length-$l$ walks ($d^l$), and $\mathbf 1^\top A_\times^l\mathbf 1 = (\mathbf 1^\top A^l\mathbf 1)(\mathbf 1^\top A'^l\mathbf 1)$
for unlabelled graphs. So for $C_3\cup C_3$ and $C_6$, which have identical walk counts at every length, the normalised kernel is exactly 1.
Walk-based kernels are blind to the same symmetric cases as 1-WL; graphlets (triangles!) are not. Labels help only when they break the symmetry.

```python
def rw_kernel(A1, A2, lam, l1=None, l2=None):
    Ax = np.kron(A1, A2)
    m = np.ones(len(Ax))
    if l1 is not None:
        m = (np.asarray(l1)[:, None] == np.asarray(l2)[None, :]).ravel().astype(float)
        Ax = Ax * np.outer(m, m)
    rho = np.max(np.abs(np.linalg.eigvals(Ax))) if Ax.any() else 0
    if rho > 0 and lam >= 1 / rho:
        raise ValueError(f'lambda must be < 1/rho = {1 / rho:.3f}')
    return m @ np.linalg.solve(np.eye(len(Ax)) - lam * Ax, m)

rw_paw_p4 = rw_kernel(PAW, P4, 0.1)
k12 = rw_kernel(two_triangles, C6, 0.1)
rw_fail = k12 / np.sqrt(rw_kernel(two_triangles, two_triangles, 0.1) * rw_kernel(C6, C6, 0.1))
print(rw_paw_p4, rw_fail)
```

</details>

### Exercise 16 · Kernel SVM for graph classification
*💻 Code · ★★★*

Two classes of 30-node graphs with the **same number of edges** (60): class 0 = small-world graphs (`small_world`, a ring lattice where
every node links to its 2 nearest neighbours on each side, 10% of edges rewired), class 1 = Erdős–Rényi graphs with 60 random edges.

Compute normalised WL Gram matrices for $h = 0, 1, 2, 3$ and evaluate `SVC(kernel='precomputed', C=1)` with 5-fold stratified CV
(`StratifiedKFold(5, shuffle=True, random_state=0)`). Store the mean accuracies in `svm_acc` (dict over $h$). Why is $h = 0$ useless here?

In [ ]:
def small_world(n, k, p, r):
    A = np.zeros((n, n))
    for i in range(n):
        for j in range(1, k + 1):
            A[i, (i + j) % n] = A[(i + j) % n, i] = 1
    for i, j in np.argwhere(np.triu(A)):          # rewire each edge with probability p
        if r.random() < p:
            free = np.flatnonzero((A[i] == 0) & (np.arange(n) != i))
            nj = r.choice(free)
            A[i, j] = A[j, i] = 0; A[i, nj] = A[nj, i] = 1
    return A

def er_m(n, m, r):
    iu, ju = np.triu_indices(n, 1); pick = r.choice(len(iu), m, replace=False)
    A = np.zeros((n, n)); A[iu[pick], ju[pick]] = 1; return A + A.T

r = np.random.default_rng(0)
graph_set = [small_world(30, 2, 0.1, r) if i % 2 == 0 else er_m(30, 60, r) for i in range(80)]
y_set = np.arange(80) % 2

svm_acc = None

if svm_acc is not None:
    check('h=0 is at chance (all graphs look alike)', abs(svm_acc[0] - 0.5) < 0.1, True)
    check('h=2 accuracy > 0.9', svm_acc[2] > 0.9, True)
    _Kn = wl_kernel(graph_set[:6], 2, normalize=True)
    check('your WL Gram is symmetric PSD', np.allclose(_Kn, _Kn.T) and np.linalg.eigvalsh(_Kn)[0] > -1e-9, True)
else:
    check('kernel SVM', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

`cross_val_score(SVC(kernel='precomputed'), K, y, cv=cv)` slices the Gram matrix rows and columns for you.

</details>

<details>
<summary><b>✅ Solution</b></summary>

At $h = 0$ every graph is "30 nodes of colour 1", so all normalised similarities are 1 and the SVM can't do better than chance.
From $h=1$ the colours encode degrees: the lattice is almost 4-regular while ER degrees are Poisson-spread, which is already
enough for 100% CV accuracy here. Deeper subtrees add the lattice's local structure (useful when degrees are matched, e.g. if you rewired more). This is the graph-level analogue of Lecture 11's message passing +
pooling, but with a fixed, injective feature map and a convex SVM instead of learnt weights.

```python
def small_world(n, k, p, r):
    A = np.zeros((n, n))
    for i in range(n):
        for j in range(1, k + 1):
            A[i, (i + j) % n] = A[(i + j) % n, i] = 1
    for i, j in np.argwhere(np.triu(A)):
        if r.random() < p:
            free = np.flatnonzero((A[i] == 0) & (np.arange(n) != i))
            nj = r.choice(free)
            A[i, j] = A[j, i] = 0; A[i, nj] = A[nj, i] = 1
    return A

def er_m(n, m, r):
    iu, ju = np.triu_indices(n, 1); pick = r.choice(len(iu), m, replace=False)
    A = np.zeros((n, n)); A[iu[pick], ju[pick]] = 1; return A + A.T

r = np.random.default_rng(0)
graph_set = [small_world(30, 2, 0.1, r) if i % 2 == 0 else er_m(30, 60, r) for i in range(80)]
y_set = np.arange(80) % 2

cv = StratifiedKFold(5, shuffle=True, random_state=0)
svm_acc = {}
for h in (0, 1, 2, 3):
    K = wl_kernel(graph_set, h, normalize=True)
    svm_acc[h] = cross_val_score(SVC(kernel='precomputed', C=1), K, y_set, cv=cv).mean()
print(svm_acc)
```

</details>

### Exercise 17 · Over-specific features at large depth
*💻 Code · ★★☆*

Using `graph_set` from the previous exercise, compute the normalised WL Gram matrix for $h = 1,\dots,8$ **without** the lower-depth features,
i.e. using only the colour histogram of iteration $h$ (write a small variant, or slice the columns of `wl_features` by iteration).
Store the mean off-diagonal similarity for each $h$ in `offdiag` (list). What happens, and how does that relate to the NCI1 over-fitting story?

In [ ]:
offdiag = None

if offdiag is not None:
    check('off-diagonal similarity decreases with depth', bool(np.all(np.diff(offdiag) < 1e-12)), True)
    check('deep colours are nearly unique (h=8 mean < 0.1)', offdiag[-1] < 0.1, True)
else:
    check('off-diagonal trend', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

Easiest: run your WL loop and build a histogram from `(t, colour)` keys with `t == h` only. Off-diagonal mean: `(K.sum() - np.trace(K)) / (N * (N - 1))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The mean off-diagonal similarity falls towards 0: at depth 8 a colour encodes an entire 8-hop rooted subtree, which is essentially unique to
one node of one graph, so the Gram matrix approaches the identity. A kernel close to $I$ means "every graph is only similar to itself": the SVM can
fit any labelling (memorisation) but has nothing to generalise from, which is the drop beyond $h\approx3$–$4$ on NCI1. Summing over all depths
$0..h$ (the standard WL kernel) softens this, because the shallow colours keep contributing shared mass.

```python
def wl_kernel_depth_only(graphs, h):
    Phi = wl_features(graphs, h)
    # recover which columns belong to iteration h by recomputing with h-1 (new colours appear only at depth h)
    n_prev = wl_features(graphs, h - 1).shape[1] if h > 0 else 0
    P = Phi[:, n_prev:]
    K = P @ P.T; d = np.sqrt(np.diag(K))
    return K / np.outer(d, d)

offdiag = []
N = len(graph_set)
for h in range(1, 9):
    K = wl_kernel_depth_only(graph_set, h)
    offdiag.append((K.sum() - np.trace(K)) / (N * (N - 1)))
print(np.round(offdiag, 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).*